#Final Project - Baseline Model


# 0. Setup

## 0.1. Libraries and Helper Functions

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,AutoConfig, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0.2 Data Acquisition

### FakeNewsNet Dataset
FAKENEWSNET is an ongoing data collection project for fake news research. It consists of headlines and body texts of fake news articles based on BuzzFeed and PolitiFact. It also collects information about the social engagements of these articles from Twitter.

https://github.com/JinyanSu1/Fakenews-dataset

The orginial fake news and real news are used as HF and HR.
MF and MR are datasets created by adopting the SMP prompting technique via ChatGPT. Generated 97 and 4,084 LLM-generated fake news for PolitiFact and GossipCop, respectively.

The dataset was created based on politifact and gossipcop dataset from FakeNewsNet (https://github.com/KaiDMML/FakeNewsNet/tree/dependabot/pip/certifi-2022.12.7), where the news articles that don't contain title or description has already been excluded.

In [ ]:
# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
# Function to process FakeNewsNet data
def process_fakenewsnet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []
    label_map = {"HF": "human_fake", "HR": "human_real", "MF": "gpt_fake", "MR": "gpt_real"}

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)
        source = "gossipcop" if "gossipcop" in category else "politifact"
        label = label_map.get(category.split("_")[-1], "unknown")

        for news_id, news_content in data.items():
            if 'text' in news_content:
                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),
                    "source": source,
                    "label": label,
                })

    return pd.DataFrame(data_list)


In [ ]:
# Define dataset paths
fake_news_paths = {category: f"/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/FakeNewsNet/{category}.json"
                    for category in ["gossipcop_HF", "gossipcop_HR", "politifact_HF", "politifact_HR",
                                     "gossipcop_MF", "gossipcop_MR", "politifact_MF", "politifact_MR"]}

# Load and process the data
FakeNewsNet_df = process_fakenewsnet_data(fake_news_paths)

# Display the first 5 rows
FakeNewsNet_df.head()

 ## 0.3. Data Preparation
Perform essential preprocessing steps to prepare the fake news dataset for further analysis or modeling:
*   Missing Value Check
*   Text Cleaning, including:
  *   Converts all text to lowercase to maintain consistency
  *   Removes URLs (both http and www formats) that do not contribute to the text meaning
  *   Eliminates extra whitespace for cleaner formatting

In [ ]:
# check if there is any missing value
FakeNewsNet_df.isnull().sum()

In [ ]:
# Function to clean text
def clean_text(text):
    """Clean text by removing URLs, extra spaces and converting to lowercase"""
    # Lowercase the text
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
# Apply the text cleaning function
FakeNewsNet_df["cleaned_text"] = FakeNewsNet_df["text"].apply(clean_text)
# Display the first 5 rows
FakeNewsNet_df.head()

 ### 0.3.1 Balance the dataset
Balance the dataset by retaining only the cleaned_text and label columns. Then, sampling each label category to ensure equal representation (4181 samples each). This helps prevent model bias caused by class imbalance.

In [ ]:
# Filter the relevant columns
FakeNewsNet_df = FakeNewsNet_df[["cleaned_text", "label"]]

In [ ]:
# check each label category
FakeNewsNet_df['label'].value_counts()

In [ ]:
# balance the data
human_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_real'].sample(n=4181, random_state=42)
gpt_real_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_real'].sample(n=4181, random_state=42)
human_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'human_fake']
gpt_fake_df = FakeNewsNet_df[FakeNewsNet_df['label'] == 'gpt_fake']

### 0.3.2 Split Data into Train, Validation, and Test Sets
Split each label category into training (80%), validation (10%), and test (10%) sets using stratified sampling to preserve label distribution. The resulting subsets are then concatenated and shuffled to form the final balanced train, validation, and test datasets.

In [ ]:
# Split each label dataset into train (80%), validation (10%), and test (10%)
def split_dataset(df, label_name):
    train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
    return train, val, test

# Apply the split function to each label category
human_real_train, human_real_val, human_real_test = split_dataset(human_real_df, "human_real")
human_fake_train, human_fake_val, human_fake_test = split_dataset(human_fake_df, "human_fake")
gpt_real_train, gpt_real_val, gpt_real_test = split_dataset(gpt_real_df, "gpt_real")
gpt_fake_train, gpt_fake_val, gpt_fake_test = split_dataset(gpt_fake_df, "gpt_fake")

# Stack the train, validation, and test sets
train_df = pd.concat([human_real_train, human_fake_train, gpt_fake_train, gpt_real_train]).sample(frac=1, random_state=42)
val_df = pd.concat([human_real_val, human_fake_val, gpt_fake_val, gpt_real_val]).sample(frac=1, random_state=42)
test_df = pd.concat([human_real_test, human_fake_test, gpt_fake_test, gpt_real_test]).sample(frac=1, random_state=42)

# Print label distributions
print("Training set label distribution:")
print(train_df['label'].value_counts())
print("\nValidation set label distribution:")
print(val_df['label'].value_counts())
print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Print dataset sizes
print(f"Training: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")



# 1. Baseline Model - Bert & Bert-variant Models
Define a complete pipeline to train and evaluate transformer-based models for fake news classification. It includes:



*   Model & Tokenizer Loading: A function to load pre-trained models (like RoBERTa, DeBERTa, ModernBERT) and their tokenizers with custom output label size
*   Tokenization & Padding: A helper to tokenize and pad input texts, handling model-specific requirements (e.g., token type IDs for BERT/DeBERTa)
*   Dataset Class: A custom PyTorch Dataset (FakeNewsDataset) for feeding tokenized inputs and labels into the model.
*   Training & Evaluation: The train_and_evaluate function trains the model using Hugging Face’s Trainer, evaluates it on the test set. The evaluation metrics used includes:
  *   Accuracy, Precision, Recall, F1-score
  *   Classification report
  *   Confusion matrix
  *   Misclassification rate
*   Model Comparison: The pipeline is used to evaluate 6 models: ModernBERT-base, ModernBERT-large, roberta-base, roberta-large, deberta-v3-base, deberta-v3-large

## Model Setup

In [ ]:
# Tokenizer and model function
def load_model_and_tokenizer(model_name, num_labels):
    """Load tokenizer and model for a given model name."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = num_labels

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    return tokenizer, model, config.model_type


# Tokenization function
def tokenize_and_pad(texts, tokenizer, model_type="bert", max_length=512):
    """Tokenize and pad the text data, ensuring compatibility with different model architectures."""
    return_token_type_ids = model_type in ["bert", "deberta"]

    encoding = tokenizer(
        texts,
        truncation=True,  # Ensure texts longer than max_length are truncated
        padding="max_length",  # Pad texts to max_length
        max_length=max_length,
        return_tensors="pt",
        return_token_type_ids=return_token_type_ids  # Only return token_type_ids for BERT/DeBERTa
    )
    return encoding

In [ ]:
# PyTorch Dataset class
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels, use_fp16):
        self.encodings = encodings
        self.labels = labels
        self.use_fp16 = use_fp16

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx], dtype=torch.long) for key, val in self.encodings.items()}

        if self.use_fp16:
            item["attention_mask"] = item["attention_mask"].to(dtype=torch.float16)

        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
def train_and_evaluate(model_name, num_labels, train_df, val_df, test_df, use_fp16=False):
    """Train and evaluate the model on fake news classification."""

    # Load tokenizer and model, and get model type
    tokenizer, model, model_type = load_model_and_tokenizer(model_name, num_labels)

    # Tokenization (pass model_type)
    train_encodings = tokenize_and_pad(train_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    val_encodings = tokenize_and_pad(val_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)
    test_encodings = tokenize_and_pad(test_df['cleaned_text'].tolist(), tokenizer, model_type=model_type)

    # Label Mapping
    label_map = {"human_real": 0, "human_fake": 1, "gpt_real": 2, "gpt_fake": 3}
    train_labels = train_df['label'].map(label_map).tolist()
    val_labels = val_df['label'].map(label_map).tolist()
    test_labels = test_df['label'].map(label_map).tolist()

    # Create Dataset
    train_dataset = FakeNewsDataset(train_encodings, train_labels, use_fp16)
    val_dataset = FakeNewsDataset(val_encodings, val_labels, use_fp16)
    test_dataset = FakeNewsDataset(test_encodings, test_labels, use_fp16)

    # Training Arguments
    training_args = TrainingArguments(
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        fp16=use_fp16,
        label_smoothing_factor=0.1
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train
    trainer.train()

    # Save Model
    model_path = f"/content/drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/{model_name}_base_model"
    trainer.save_model(model_path)

    # Evaluate
    eval_results = trainer.evaluate(test_dataset)
    print(f"Evaluation Results for {model_name}: {eval_results}")

    # Predict
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)

    # Classification Report
    report = classification_report(test_labels, y_pred, target_names=list(label_map.keys()))
    print(f"Classification Report for {model_name}:\n{report}")

    # Confusion Matrix
    conf_matrix = confusion_matrix(test_labels, y_pred)
    print(f"Confusion Matrix for {model_name}:\n{conf_matrix}")

    # Misclassification Rate
    misclassification_rate = 1 - accuracy_score(test_labels, y_pred)
    print(f"Misclassification Rate for {model_name}: {misclassification_rate:.4f}")


In [ ]:
# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Model Training

In [ ]:
# Train and evaluate RoBERTa, BERT, DeBERTa, and ModernBERT model
train_and_evaluate("answerdotai/ModernBERT-base", 4, train_df, val_df, test_df, use_fp16=True)

In [ ]:
# Train and evaluate RoBERTa, BERT, DeBERTa, and ModernBERT model
train_and_evaluate("answerdotai/ModernBERT-large", 4, train_df, val_df, test_df, use_fp16=True)

In [ ]:
train_and_evaluate("roberta-base", 4, train_df, val_df, test_df, use_fp16=True)

In [ ]:
train_and_evaluate("roberta-large", 4, train_df, val_df, test_df, use_fp16=True)

In [ ]:
train_and_evaluate("microsoft/deberta-v3-base", 4, train_df, val_df, test_df, use_fp16=False);

In [ ]:
train_and_evaluate("microsoft/deberta-v3-large", 4, train_df, val_df, test_df, use_fp16=False);

In [ ]:
train_and_evaluate("bert-base-uncased", num_labels=4, train_df=train_df, val_df=val_df, test_df=test_df)

In [ ]:
train_and_evaluate("bert-large-uncased", num_labels=4, train_df=train_df, val_df=val_df, test_df=test_df)